# Sestoji - Data Exploration
This notebook lets you inspect the `sestoji_processed.csv` file column by column.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

df = pd.read_csv('sestoji_processed.csv')

print(f'Total rows: {len(df)}')
print(f'Total columns: {len(df.columns)}')
print(f'\nColumns: {list(df.columns)}')

Total rows: 350593
Total columns: 28

Columns: ['ggo', 'odsek', 'sestoj', 'povrsina', 'rfaza', 'lzskdv11', 'lzskdv11_m', 'lzskdv21', 'lzskdv21_m', 'lzskdv30', 'lzskdv34', 'lzskdv39', 'lzskdv41', 'lzskdv50', 'lzskdv60', 'lzskdv70', 'lzskdv80', 'sklep', 'zasnova', 'negovan', 'pompov', 'pomzas', 'lzigl', 'lzlst', 'lzsku', 'etigl', 'etlst', 'etsku']


/tmp/ipykernel_14586/744627355.py:5: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('sestoji_processed.csv')


## Code → Naziv lookup tables
Loads the original `.gpkg` to build lookup tables so you can see what each numeric code means.
These are for inspection only — not added to the processed CSV.

In [ ]:
gdf_raw = gpd.read_file('sestoji.gpkg', ignore_geometry=True)

lookup_pairs = [
    ('rfaza',    'rfaza_naziv'),
    ('sklep',    'sklep_naziv'),
    ('zasnova',  'zasnova_naziv'),
    ('negovan',  'negovanost_naziv'),
    ('pomzas',   'pomzas_naziv'),
    ('sksmerni', 'sksmerni_naziv'),
]

lookups = {}
for code_col, naziv_col in lookup_pairs:
    if code_col in gdf_raw.columns and naziv_col in gdf_raw.columns:
        lookup = gdf_raw[[code_col, naziv_col]].drop_duplicates().dropna().sort_values(code_col)
        lookup.columns = ['code', 'naziv']
        lookups[code_col] = lookup
        print(f'\n--- {code_col} ---')
        display(lookup.reset_index(drop=True))
    else:
        print(f'Skipping {code_col} / {naziv_col} — not found in gpkg')

## Decoded view — first 100 rows with naziv labels
Shows the processed data with a decoded label column inserted next to each coded column.

In [ ]:
df_decoded = df.copy()

for code_col, lookup in lookups.items():
    if code_col in df_decoded.columns:
        mapping = lookup.set_index('code')['naziv'].to_dict()
        idx = df_decoded.columns.get_loc(code_col) + 1
        decoded_col = code_col + '_naziv'
        df_decoded.insert(idx, decoded_col, df_decoded[code_col].map(mapping))

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
df_decoded.head(100)

## First 100 rows — processed CSV only (no labels)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df.head(100)

## Null counts per column

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_%': null_pct})
null_summary[null_summary['null_count'] > 0].sort_values('null_%', ascending=False)

## Basic statistics — numeric columns

In [ ]:
df.describe()

## Basic statistics — categorical columns

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().head(20))

## Distribution plots — numeric columns
One histogram per numeric column.

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()

for col in num_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    data = df[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='none')
    ax.set_title(f'{col}  (nulls: {df[col].isnull().sum()}, unique: {df[col].nunique()})')
    ax.set_xlabel(col)
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()

## Inspect a specific column in detail
Change `col_name` to whichever column you want to look at more closely.

In [ ]:
col_name = 'rfaza'  # <-- change this

print(f'Column: {col_name}')
print(f'Type: {df[col_name].dtype}')
print(f'Nulls: {df[col_name].isnull().sum()} ({df[col_name].isnull().mean()*100:.1f}%)')
print(f'Unique values: {df[col_name].nunique()}')
print(f'\nFirst 50 values:')
print(df[col_name].head(50).to_string())
print(f'\nValue counts (top 30):')
if col_name in lookups:
    mapping = lookups[col_name].set_index('code')['naziv'].to_dict()
    vc = df[col_name].value_counts().head(30).reset_index()
    vc.columns = ['code', 'count']
    vc['naziv'] = vc['code'].map(mapping)
    display(vc)
else:
    print(df[col_name].value_counts().head(30))

## Species composition columns side by side
All lzskdv columns together for a quick overview.

In [ ]:
species_cols = [c for c in df.columns if c.startswith('lzskdv')]
if species_cols:
    print('Species composition columns summary:')
    display(df[species_cols].describe())
    print(f'\nFirst 50 rows:')
    display(df[species_cols].head(50))
else:
    print('No lzskdv columns found.')

## Timber stock columns side by side
lzigl, lzlst, lzsku, etigl, etlst, etsku

In [ ]:
stock_cols = [c for c in ['lzigl', 'lzlst', 'lzsku', 'etigl', 'etlst', 'etsku'] if c in df.columns]
if stock_cols:
    print('Timber stock columns summary:')
    display(df[stock_cols].describe())
    print(f'\nFirst 50 rows:')
    display(df[stock_cols].head(50))
else:
    print('No timber stock columns found.')

## Smreka (spruce) focused view
Rows sorted by smreka percentage descending.

In [ ]:
if 'lzskdv11' in df.columns:
    spruce_view = df.sort_values('lzskdv11', ascending=False)
    print('Top 50 sestoji by spruce percentage:')
    display(spruce_view.head(50))
else:
    print('lzskdv11 column not found.')

## Rows per odsek
How many sestoji exist per odsek.

In [ ]:
if 'odsek' in df.columns:
    sestoji_per_odsek = df.groupby('odsek').size().reset_index(name='sestoj_count')
    print(f'Total unique odseki: {sestoji_per_odsek["odsek"].nunique()}')
    print(f'\nSestoji per odsek stats:')
    print(sestoji_per_odsek['sestoj_count'].describe())
    print(f'\nFirst 50 odseki:')
    display(sestoji_per_odsek.head(50))

## Filter rows by condition
Example: show sestoji where smreka percentage is above a threshold.

In [ ]:
# Example filter — adjust as needed
if 'lzskdv11' in df.columns:
    filtered = df[df['lzskdv11'] > 50]
    print(f'Rows with lzskdv11 (smreka) > 50%: {len(filtered)}')
    display(filtered.head(50))